In [10]:
import json

from torch.fx.experimental.symbolic_shapes import create_contiguous

with open("/Users/nad/mobiraph/data/n13_repbase_processed/hierarchy_sequences_02_ltr_correction_with_classes.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(type(metadata))

<class 'dict'>


In [11]:
general_dict = {}

In [12]:
for class_name, class_info in metadata.items():
    sequences = class_info["sequences"]
    for sequence in sequences:
        general_dict[sequence] = {"class": class_name}

In [17]:
for class_name, class_info in metadata.items():
    if class_name == 'Class II (DNA transposons)':
        sequences = class_info["sequences"]
        for sequence in sequences:
            general_dict[sequence]["order"] = "DNA transposon"

In [19]:
general_dict['MARINER62_CB']

{'class': 'Class II (DNA transposons)',
 'superfamily': 'Mariner/Tc1',
 'order': 'DNA transposon'}

In [14]:
for supfam_name, supfam_info in metadata['Class II (DNA transposons)']["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [23]:
for order_name, order_info in metadata['Class I (Retrotransposons)']["subs"].items():
    sequences = order_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["order"] = order_name

In [24]:
general_dict['LTR4_CR-LTR']

{'class': 'Class I (Retrotransposons)', 'order': 'LTR Retrotransposon'}

In [25]:
for supfam_name, supfam_info in metadata['Class I (Retrotransposons)']["subs"]["LTR Retrotransposon"]["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [29]:
general_dict['GYPSY1-LTR_CB']

{'class': 'Class I (Retrotransposons)',
 'order': 'LTR Retrotransposon',
 'superfamily': 'Gypsy'}

In [30]:
for supfam_name, supfam_info in metadata['Class I (Retrotransposons)']["subs"]["Non-LTR Retrotransposon"]["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [32]:
general_dict['SINEX-1_CR']

{'class': 'Class I (Retrotransposons)',
 'order': 'Non-LTR Retrotransposon',
 'superfamily': 'SINE'}

In [36]:
with open("/Users/nad/mobiraph/data/n13_repbase_processed/metadata_03.json", "w", encoding="utf-8") as f:
    json.dump(general_dict, f, ensure_ascii=False, indent=4)

In [54]:
sv_plants_category = {}

with open("/Users/nad/mobiraph/data/plant_sv_fam_orf_on_repbase_best.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_plants_category[parts[0]] = general_dict[parts[1]]

In [58]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_plants_category.json", "w", encoding="utf-8") as f:
    json.dump(sv_plants_category, f, ensure_ascii=False, indent=4)

In [55]:
len(sv_plants_category)

11914

In [56]:
sv_insects_category = {}

with open("/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_insects_category[parts[0]] = general_dict[parts[1]]

In [59]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_insects_category.json", "w", encoding="utf-8") as f:
    json.dump(sv_insects_category, f, ensure_ascii=False, indent=4)

In [60]:
len(sv_insects_category)

7773

In [92]:
def create_hierarchy_dict(g_dict):
    hierarchy_sequences_sv = {}
    # class level
    hierarchy_sequences_sv['Class I (Retrotransposons)'] = {'sequences': [], 'subs': {}}
    hierarchy_sequences_sv['Class II (DNA transposons)'] = {'sequences': [], 'subs': {}}
    for name, info in g_dict.items():
        hierarchy_sequences_sv[info['class']]['sequences'].append(name)
    # order level
    hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['LTR Retrotransposon'] = {'sequences': [], 'subs': {}}
    hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon'] = {'sequences': [], 'subs': {}}
    for name, info in g_dict.items():
        if not info['order']:
            continue
        if info['class'] == 'Class I (Retrotransposons)':
            hierarchy_sequences_sv['Class I (Retrotransposons)']['subs'][info['order']]['sequences'].append(name)
    # superfamily level
    for superfamily in metadata['Class I (Retrotransposons)']['subs']['LTR Retrotransposon']['subs'].keys():
        hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['LTR Retrotransposon']['subs'][superfamily] = {'sequences': [], 'subs': {}}
    for superfamily in metadata['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon']['subs'].keys():
        hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon']['subs'][superfamily] = {'sequences': [], 'subs': {}}
    for superfamily in metadata['Class II (DNA transposons)']['subs'].keys():
        hierarchy_sequences_sv['Class II (DNA transposons)']['subs'][superfamily] = {'sequences': [], 'subs': {}}

    for name, info in g_dict.items():
        if 'superfamily' not in info:
            continue
        if info['class'] == 'Class I (Retrotransposons)':
            hierarchy_sequences_sv['Class I (Retrotransposons)']['subs'][info['order']]['subs'][info['superfamily']]['sequences'].append(name)
        if info['class'] == 'Class II (DNA transposons)':
            hierarchy_sequences_sv['Class II (DNA transposons)']['subs'][info['superfamily']]['sequences'].append(name)
    return hierarchy_sequences_sv


In [93]:
hierarchy_sequences_sv_plants = create_hierarchy_dict(sv_plants_category)
hierarchy_sequences_sv_insects = create_hierarchy_dict(sv_insects_category)

In [95]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_plants.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_sv_plants, f, ensure_ascii=False, indent=4)
with open("/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_insects.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_sv_insects, f, ensure_ascii=False, indent=4)

In [99]:
sv_plants_category["plant_phaseolus_vulgaris|SVgr_3_id_085775|7909"]

{'class': 'Class I (Retrotransposons)',
 'order': 'Non-LTR Retrotransposon',
 'superfamily': 'L1'}

In [106]:
files = []
for i in range(10):
    files.append(f"/Users/nad/NeuralTE/plants_output/domain/{i}.out")

with open("/Users/nad/NeuralTE/plants_output/domain/all.out", "w", encoding="utf-8") as outfile:
    for fname in files:
        with open(fname, "r", encoding="utf-8") as infile:
            outfile.write(infile.read())

In [107]:
neuralte_results = {}

with open("/Users/nad/NeuralTE/plants_output/domain/all.out") as f:
    for line in f:
        parts = line.strip().split("\t")

        name = parts[0]
        type_full = parts[1]

        type_clean = type_full.split("#")[-1]

        neuralte_results[name] = type_clean

print(neuralte_results["plant_oryza_meridionalis|SVgr_11_id_16072|27066"])

LTR/Gypsy


In [115]:
len(set(neuralte_results.values()))

24

In [110]:
all_count = 0
true_count = 0

def has_common_substring(s1, s2, min_len=2):
    for i in range(len(s1) - min_len + 1):
        sub = s1[i:i+min_len]
        if sub in s2:
            return True
    return False


for name in neuralte_results.keys():
    if name not in sv_plants_category:
        continue
    all_count += 1
    # if 'superfamily' not in sv_plants_category[name].keys():
    #     if has_common_substring(neuralte_results[name],
    #                         sv_plants_category[name]['order']):
    #         true_count += 1
    #     else:
    #         print(neuralte_results[name], sv_plants_category[name]['order'])
    if 'superfamily' not in sv_plants_category[name].keys():
        continue
    else:
        if has_common_substring(neuralte_results[name],
                            sv_plants_category[name]['superfamily']):
            true_count += 1
        else:
            print(neuralte_results[name], sv_plants_category[name]['order'])

DNA/hAT-Ac LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LINE/L1 DNA transposon
LTR/Copia LTR Retrotransposon
LTR/Copia DNA transposon
LTR/Gypsy LTR Retrotransposon
LINE/L1 Non-LTR Retrotransposon
DNA/MULE-MuDR DNA transposon
DNA/MULE-MuDR LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
LTR/Gypsy DNA transposon
LTR/Caulimovirus DNA transposon
LINE/L1 DNA transposon
LTR/Gypsy LTR Retrotransposon
LTR/Gypsy Non-LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
DNA/CMC-EnSpm DNA transposon
DNA/CMC-EnSpm LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
DNA/PIF-Harbinger, LTR Retrotransposon
DNA/PIF-Harbinger LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Gypsy Non-LTR Retrotransposon
DNA/Ginger-1 LTR Retrotransposon
LTR/Copia DNA transpos

In [111]:
true_count / all_count

0.9863599893019523

In [112]:
true_count, all_count

(11064, 11217)

In [122]:
import csv

neuralte_results = {}

with open("/Users/nad/NeuralTE/plants_output/classified.info", newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        neuralte_results[row["#Seq_Name"]] = row["Predict_Label"]

print(neuralte_results['plant_rubus_idaeus|SVgr_6_id_059504|5905'])

Copia


In [126]:
all_count = 0
true_count = 0

for name in neuralte_results.keys():
    if name not in sv_plants_category:
        continue
    all_count += 1
    if 'superfamily' not in sv_plants_category[name].keys():
        continue
    else:
        if has_common_substring(neuralte_results[name], sv_plants_category[name]['superfamily']):
            print("same", neuralte_results[name], sv_plants_category[name]['superfamily'])
            true_count += 1
        else:
            print(neuralte_results[name], sv_plants_category[name]['superfamily'])

same Gypsy Gypsy
same Gypsy Gypsy
same Gypsy Gypsy
same Gypsy Gypsy
same Copia Copia
same L1 L1
same Copia Copia
same Gypsy Gypsy
same Copia Copia
same L1 L1
same Copia Copia
same Copia Copia
same Copia Copia
same Copia Copia
same Gypsy Gypsy
same L1 L1
same Copia Copia
same Mutator MuDR
same L1 L1
same Gypsy Gypsy
same Gypsy Gypsy
same Gypsy Gypsy
same Copia Copia
same PIF-Harbinger Harbinger
same Mutator MuDR
same Copia Copia
same Gypsy Gypsy
same Gypsy Gypsy
same Copia Copia
tRNA Gypsy
same L1 L1
same Copia Copia
same Copia Copia
same L1 L1
same L1 L1
same Gypsy Gypsy
same L1 L1
same Gypsy Gypsy
Tc1-Mariner L1
same Copia Copia
same Copia Copia
same L1 L1
same Gypsy Gypsy
same L1 L1
same Gypsy Gypsy
same Gypsy Gypsy
same Copia Copia
same Gypsy Gypsy
same Copia Copia
same Copia Copia
same Copia Copia
same Copia Copia
same L1 L1
same Copia Copia
same Copia Copia
same Copia Copia
same Copia Copia
same Gypsy Gypsy
same Copia Copia
same Copia Copia
same Gypsy Gypsy
same hAT hAT
same Copia

In [125]:
print(true_count / all_count)

0.9372167198254154


# Предсказания на кусочках

In [116]:
import pandas as pd


def load_name_to_class(csv_path: str) -> dict[str, str]:
    df = pd.read_csv(csv_path)

    if "name" not in df.columns or "y_pred" not in df.columns:
        raise ValueError("Ожидаются колонки 'name' и 'y_pred'")

    return dict(zip(df["name"], df["y_pred"]))


# пример
name_to_pred_class = load_name_to_class(
    f"/Users/nad/mobiraph/data/n29_sv_insects_results_30/root/ensemble.csv"
)

In [143]:
import pandas as pd

HIERARCHY_ROOTS = [
    "root",
    "Class I (Retrotransposons)",
    "Class II (DNA transposons)",
    "Class I (Retrotransposons)\tLTR Retrotransposon",
    "Class I (Retrotransposons)\tNon-LTR Retrotransposon",
]

# загружаем заранее
name_to_pred_superfamily_class2 = load_name_to_class(
    "/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class II (DNA transposons)/ensemble.csv"
)

name_to_pred_order_class1 = load_name_to_class(
    "/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class I (Retrotransposons)/ensemble.csv"
)

name_to_pred_superfamily_ltr = load_name_to_class("/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class I (Retrotransposons)\tLTR Retrotransposon/ensemble.csv")
name_to_pred_superfamily_nonltr = load_name_to_class("/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class I (Retrotransposons)\tNon-LTR Retrotransposon/ensemble.csv")

result = {}

for name, class_name in name_to_pred_class.items():
    if class_name == "Class II (DNA transposons)":
        result[name] = name_to_pred_superfamily_class2.get(name)
    else:
        result[name] = name_to_pred_order_class1.get(name)

for name, class_name in result.items():
    if class_name == 'LTR Retrotransposon':
        result[name] = name_to_pred_superfamily_ltr.get(name)
    elif class_name == 'Non-LTR Retrotransposon':
        result[name] = name_to_pred_superfamily_nonltr.get(name)
    else:
        continue

result

{'name1': 'RTE',
 'name2': 'Non-LTR Retrotransposon_other',
 'name3': 'BEL',
 'name4': 'Gypsy',
 'name5': 'Gypsy',
 'name6': 'Gypsy',
 'name7': 'L1',
 'name8': 'BEL',
 'name9': 'Non-LTR Retrotransposon_other',
 'name10': 'Non-LTR Retrotransposon_other',
 'name11': 'Gypsy',
 'name12': 'BEL',
 'name13': 'Gypsy',
 'name14': 'Gypsy',
 'name15': 'Non-LTR Retrotransposon_other',
 'name16': 'Gypsy',
 'name17': 'RTE',
 'name18': 'Non-LTR Retrotransposon_other',
 'name19': 'Gypsy',
 'name20': 'Gypsy',
 'name21': 'Non-LTR Retrotransposon_other',
 'name22': 'DIRS',
 'name23': 'Non-LTR Retrotransposon_other',
 'name24': 'CR1',
 'name25': 'Gypsy',
 'name26': 'Gypsy',
 'name27': 'Gypsy',
 'name28': 'Gypsy',
 'name29': 'Gypsy',
 'name30': 'BEL',
 'name31': 'Non-LTR Retrotransposon_other',
 'name32': 'Gypsy',
 'name33': 'Gypsy',
 'name34': 'CR1',
 'name35': 'Non-LTR Retrotransposon_other',
 'name36': 'Gypsy',
 'name37': 'Non-LTR Retrotransposon_other',
 'name38': 'CR1',
 'name39': 'Non-LTR Retrotransp

In [167]:
with open("result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

In [ ]:
/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best_30.txt

In [136]:
sv_insects_category_30 = {}

with open("/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best_30.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_insects_category_30[parts[0]] = general_dict[parts[1]]

In [ ]:
HIERARCHY_ROOTS = [
    "root",
    "Class I (Retrotransposons)",
    "Class II (DNA transposons)",
    "Class I (Retrotransposons)\tLTR Retrotransposon",
    "Class I (Retrotransposons)\tNon-LTR Retrotransposon",
]


In [144]:
sv_insects_category_30_superfamily = {}
for name, info in sv_insects_category_30.items():
    if 'superfamily' not in info:
        continue
    sv_insects_category_30_superfamily[name] = info['superfamily']

In [168]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for name in sv_insects_category_30_superfamily.keys():
    y_true.append(sv_insects_category_30_superfamily[name])
    y_pred.append(result[name])

print(classification_report(y_true, y_pred))

/Users/nad/miniconda3/envs/mobiraph/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nad/miniconda3/envs/mobiraph/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/nad/miniconda3/envs/mobiraph/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

In [165]:
import csv

neuralte_results_30 = {}

with open("/Users/nad/NeuralTE/insects_output_30/classified.info", newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        neuralte_results_30[row["#Seq_Name"]] = row["Predict_Label"]

print(neuralte_results_30['name1'])

Tc1-Mariner


In [166]:
set(neuralte_results_30.values())

{'Bel-Pao',
 'CACTA',
 'Copia',
 'Crypton',
 'DIRS',
 'Gypsy',
 'Helitron',
 'I',
 'Jockey',
 'L1',
 'Merlin',
 'Mutator',
 'P',
 'PIF-Harbinger',
 'Penelope',
 'R2',
 'RTE',
 'Retrovirus',
 'Tc1-Mariner',
 'Transib',
 'Unknown',
 'hAT',
 'tRNA'}

In [134]:
sv_insects_category_30

{'name1': 'Class I (Retrotransposons)',
 'name2': 'Class II (DNA transposons)',
 'name3': 'Class I (Retrotransposons)',
 'name4': 'Class I (Retrotransposons)',
 'name6': 'Class I (Retrotransposons)',
 'name7': 'Class I (Retrotransposons)',
 'name8': 'Class I (Retrotransposons)',
 'name9': 'Class I (Retrotransposons)',
 'name10': 'Class I (Retrotransposons)',
 'name11': 'Class I (Retrotransposons)',
 'name12': 'Class II (DNA transposons)',
 'name13': 'Class I (Retrotransposons)',
 'name14': 'Class I (Retrotransposons)',
 'name15': 'Class I (Retrotransposons)',
 'name16': 'Class I (Retrotransposons)',
 'name17': 'Class I (Retrotransposons)',
 'name18': 'Class I (Retrotransposons)',
 'name19': 'Class I (Retrotransposons)',
 'name20': 'Class I (Retrotransposons)',
 'name21': 'Class I (Retrotransposons)',
 'name22': 'Class I (Retrotransposons)',
 'name23': 'Class II (DNA transposons)',
 'name24': 'Class I (Retrotransposons)',
 'name25': 'Class I (Retrotransposons)',
 'name26': 'Class I (Ret

In [157]:
superfamily_to_class = {
    # Class 2 — DNA transposons
    "Mariner/Tc1": "Class 2",
    "hAT": "Class 2",
    "MuDR": "Class 2",
    "EnSpm/CACTA": "Class 2",
    "piggyBac": "Class 2",
    "Harbinger": "Class 2",
    "Helitron": "Class 2",
    "Kolobok": "Class 2",
    "Academ": "Class 2",
    "DNA transposon_other": "Class 2",

    # Class 1 — LTR retrotransposons
    "Gypsy": "Class 1",
    "Copia": "Class 1",
    "BEL": "Class 1",
    "DIRS": "Class 1",
    "Troyka": "Class 1",

    # Class 1 — Non-LTR retrotransposons
    "SINE": "Class 1",
    "L1": "Class 1",
    "RTE": "Class 1",
    "CR1": "Class 1",
    "Tx1": "Class 1",
    "RTEX": "Class 1",
    "Tad1": "Class 1",
    "Non-LTR Retrotransposon_other": "Class 1"
}

In [153]:
set(neuralte_results_30.values())

{'Bel-Pao',
 'CACTA',
 'Copia',
 'Crypton',
 'DIRS',
 'Gypsy',
 'Helitron',
 'I',
 'Jockey',
 'L1',
 'Merlin',
 'Mutator',
 'P',
 'PIF-Harbinger',
 'Penelope',
 'R2',
 'RTE',
 'Retrovirus',
 'Tc1-Mariner',
 'Transib',
 'Unknown',
 'hAT',
 'tRNA'}

In [154]:
superfamily_to_class.keys()

dict_keys(['Mariner/Tc1', 'hAT', 'MuDR', 'EnSpm/CACTA', 'piggyBac', 'Harbinger', 'Helitron', 'Kolobok', 'Academ', 'DNA_transposon_other', 'Gypsy', 'Copia', 'BEL', 'DIRS', 'Troyka', 'SINE', 'L1', 'RTE', 'CR1', 'Tx1', 'RTEX', 'Tad1', 'Non-LTR_Retrotransposon_other'])

In [161]:
mapping = {
    'Bel-Pao': 'BEL',
    'CACTA': 'EnSpm/CACTA',
    'Copia': 'Copia',
    'Crypton': 'DNA transposon_other',
    'DIRS': 'DIRS',
    'Gypsy': 'Gypsy',
    'Helitron': 'Helitron',
    'I': 'Non-LTR Retrotransposon_other',
    'Jockey': 'Non-LTR Retrotransposon_other',
    'L1': 'L1',
    'Merlin': 'DNA transposon_other',
    'Mutator': 'MuDR',
    'P': 'DNA transposon_other',
    'PIF-Harbinger': 'Harbinger',
    'Penelope': 'Non-LTR Retrotransposon_other',
    'R2': 'Non-LTR Retrotransposon_other',
    'RTE': 'RTE',
    'Retrovirus': 'Gypsy',
    'Tc1-Mariner': 'Mariner/Tc1',
    'Transib': 'DNA transposon_other',
    'Unknown': 'DNA transposon_other',
    'hAT': 'hAT',
    'tRNA': 'SINE'
}

In [173]:
from sklearn.metrics import classification_report

true_count = 0
all_count = 0

y_true = []
y_pred = []

for name in neuralte_results_30.keys():
    if name not in sv_insects_category_30:
        continue

    if 'superfamily' not in sv_insects_category_30[name]:
        continue

    pred = superfamily_to_class[mapping[neuralte_results_30[name]]]
    true = superfamily_to_class[sv_insects_category_30[name]['superfamily']]

    all_count += 1

    y_true.append(true)
    y_pred.append(pred)

    if pred == true:
        true_count += 1

print(true_count / all_count if all_count > 0 else 0)
print(true_count, all_count)

# classification report
print(classification_report(y_true, y_pred))

0.6738009283135636
5226 7756
              precision    recall  f1-score   support

     Class 1       0.99      0.60      0.75      6284
     Class 2       0.36      0.97      0.53      1472

    accuracy                           0.67      7756
   macro avg       0.68      0.79      0.64      7756
weighted avg       0.87      0.67      0.71      7756



In [163]:
print(true_count / all_count)

0.672327286761868


In [141]:
true_count, all_count

(4420, 7773)